# 02 — Training

The complete pipeline for training your own LPF model:

1. Generate synthetic data
2. Train the LPF encoder + decoder **(also indexes train/val evidence automatically)**
3. Train the aggregator *(uses the index from Step 2)*
4. Index test evidence *(needed before inference/evaluate)*

> **Shortcut:** `epalea train-full` runs Steps 2 and 3 together. You still run Step 4 separately.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "epalea", "-q"])
import epalea
print(f"epalea {epalea.__version__} ready")

## Stage 0 — Generate synthetic compliance data

In [ ]:
import subprocess

# Generate compliance data matching the pretrained model parameters
result = subprocess.run([
    "epalea", "generate-data",
    "--domain", "compliance",
    "--n-entities", "300",
    "--years", "2020", "--years", "2021", "--years", "2022",
    "--evidence-per-entity", "5",
    "--noise-level", "0.1",
    "--contradictory-rate", "0.05",
    "--output-dir", "./user_workspace/data",
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


In [ ]:
import json
from pathlib import Path

data_dir = Path("./user_workspace/data/compliance")
for fname in sorted(data_dir.glob("*.json")):
    with open(fname) as f:
        items = json.load(f)
    print(f"  {fname.name:<35} {len(items):>5} records")


## Step 2 — Train the LPF encoder + decoder

Seed search tests 7 seeds and saves the best checkpoint to `./user_workspace/checkpoints/compliance/best_model.pt`.

**This step also automatically indexes train and val evidence.** You will find `vector_store.faiss` and `metadata.jsonl` in `./user_workspace/data/compliance/` after training completes.


In [ ]:
result = subprocess.run([
    "epalea", "train",
    "--domain", "compliance",
    "--predicate", "compliance_level",
    "--domain-values", "low", "--domain-values", "medium", "--domain-values", "high",
    "--data-dir", "./user_workspace/data/compliance",
    "--checkpoint-dir", "./user_workspace/checkpoints/compliance",
], capture_output=True, text=True)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
from pathlib import Path
import json

chk_dir = Path("./user_workspace/checkpoints/compliance")
print("Checkpoint directory contents:")
for f in sorted(chk_dir.iterdir()):
    print(f"  {f.name}")

data_dir = Path("./user_workspace/data/compliance")
print("\nIndex files in data directory:")
for f in sorted(data_dir.glob("*.faiss")) + sorted(data_dir.glob("*.jsonl")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")


## Step 3 — Train the aggregator

The evidence index was built automatically in Step 2. The aggregator uses this index to learn quality and consistency weights for each evidence item.

`--index-dir` and `--data-dir` point to the same directory because `epalea train` wrote the index into the data directory.


In [ ]:
result = subprocess.run([
    "epalea", "train-aggregator",
    "--checkpoint", "./user_workspace/checkpoints/compliance/best_model.pt",
    "--schema", "./user_workspace/checkpoints/compliance/schema.json",
    "--index-dir", "./user_workspace/data/compliance",
    "--data-dir", "./user_workspace/data/compliance",
    "--aggregator-checkpoint-dir", "./user_workspace/checkpoints/compliance",
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
from pathlib import Path

chk_dir = Path("./user_workspace/checkpoints/compliance")
print("Checkpoints after aggregator training:")
for f in sorted(chk_dir.iterdir()):
    print(f"  {f.name}")


## Step 4 — Index test evidence

Test entities are not yet in the index. Run `epalea index` to add them before inference or evaluate.


In [ ]:
result = subprocess.run([
    "epalea", "index",
    "--checkpoint", "./user_workspace/checkpoints/compliance/best_model.pt",
    "--schema", "./user_workspace/checkpoints/compliance/schema.json",
    "--evidence", "./user_workspace/data/compliance/test_evidence.json",
    "--predicate", "compliance_level",
    "--index-dir", "./user_workspace/data/compliance",
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
## Step 5 — Infer and evaluate

# All assets are ready. Run inference and evaluation. Check 03_inference.Ipynb


## Alternative: train-full (Steps 2 + 3 together)

`epalea train-full` runs LPF training + aggregator training from a single config.
You still need to run Step 4 (`epalea index`) separately before inference.

```yaml
# ./user_workspace/configs/train_full.yaml
domain: compliance
predicate: compliance_level
domain_values: [low, medium, high]
data_dir: ./user_workspace/data/compliance
checkpoint_dir: ./user_workspace/checkpoints/compliance
n_seeds: 7
embedding_dim: 384
latent_dim: 64
epochs: 50
aggregator_epochs: 30
```


In [ ]:
# Write the combined config
from pathlib import Path
import yaml

config = {
    "domain": "compliance",
    "predicate": "compliance_level",
    "domain_values": ["low", "medium", "high"],
    "data_dir": "./user_workspace/data/compliance",
    "checkpoint_dir": "./user_workspace/checkpoints/compliance",
    "n_seeds": 3,           # reduced for demo speed
    "embedding_dim": 384,
    "latent_dim": 64,
    "epochs": 30,
    "aggregator_epochs": 20,
}

config_path = Path("./user_workspace/configs/train_full.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.dump(config))
print(f"Config written to {config_path}")

# Run train-full
# result = subprocess.run(
#     ["epalea", "train-full", "--config", str(config_path)],
#     capture_output=True, text=True
# )
# print(result.stdout)
print("Uncomment the lines above to run train-full.")
print("Remember to run 'epalea index' for test evidence afterwards.")
